In [1]:
from dotenv import load_dotenv

In [26]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain.tools import tool
from langchain.agents import create_agent

### Load the Data

In [5]:
loader=PyPDFLoader("../data/medical_report.pdf")

docs=loader.load()
len(docs)

9

### Split into multiplle chunks

In [6]:
spillter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splitted_data=spillter.split_documents(docs)
len(splitted_data)

26

### Converts Chunks into vector embedding and store into vector database

In [7]:
embeddings=GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")

In [10]:
vector_store=InMemoryVectorStore.from_documents(
    documents=splitted_data,
    embedding=embeddings
)

In [ ]:
# same_record=vector_store.similarity_search("patient name")
# same_record[0].page_content

### agent = tools , llm , prompt

In [ ]:
@tool
def retriver_tool(query:str):
    """
    This tool can help you to retrive the relevant data of the PDF document 
    and these pdf docuemnt have detailed about medical reports
    """
    # print("Tool calling",query)
    docs=vector_store.similarity_search(query=query,k=4)
    context=""
    for doc in docs:
        context=doc.page_content + "\n\n"

    return context
    

In [34]:
retriver_tool.invoke("patent name")

Tool calling patent name


'Report Status    \nFemale\n27 Years:\n:\n:\n:\nAge\nGender\nReported        \nP\n9/7/2025   4:56:00PM\nDR NITIN NAHAR\n474764803\nMs. NIKITA  CHUDHARY:\n:\n:\n:\n:\nName        \nLab No.    \nRef By \nCollected       \nA/c Status \n10/7/2025  6:31:50PM\nFinal\nLPL - Bhopal Lab II\nPlot No.05, Mandakini Housing Society, \nNear Apurti Shopping Mall, Kolar Main \nRoad, Bhopal, M.P. -  462042\n::Collected at            Processed at             BHOPAL CC-82\nMr Rachel V John Pata So Vitus John Mig 26 \nGraund,Indrapuri, Phone: 8770817968\n \nTest Report      \nTest Name Results Units Bio. Ref. Interval\nDecreased Levels\n· Inadequate exposure to sunlight\n· Dietary deficiency\n· Vitamin D malabsorption\n· Severe  Hepatocellular disease\n· Drugs like Anticonvulsants\n· Nephrotic syndrome\nIncreased levels\nVitamin D intoxication \nDr.Swapnil Gupta\nMD, Pathology\nChief of Laboratory                            \nDr Lal PathLabs Ltd\nDr Kiran Bhargava Pathak\nMD, Pathology\nChief of Laborator

In [35]:
llm=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [36]:
system_prompt=""" 

you are a helpful assistant that answers questions using retrieved context.
Always use the 'retriver_tool' tool for questions required external knowledge.
"""

In [37]:
agent=create_agent(
    model=llm,
    tools=[retriver_tool],
    system_prompt=system_prompt
)

In [40]:
query="what is the  name of patient and what is the name of doctors"
response=agent.invoke({"messages":[{"role":"user","content":query}]})

Tool calling patient name doctor name


In [39]:
result=response["messages"][-1].content
print(result)

[{'type': 'text', 'text': 'Based on the medical report:\n\n* **Patient Name:** Ms. Nikita Chudhary (also mentioned as Mr. Rachel V John in another section of the address/contact block)\n* **Doctor Name:** Dr. Nitin Nahar (referred by)', 'extras': {'signature': 'El4KXAERTTIPd8sQTbrww6GDxkCz/+r+Y99q7YD/Ii/fGzQUVduTfD4wp8iBAu9HVYIihDoT5K8pP8rlwxU4IzETbf7Opn17Zy40xdpB2SozHkpd6Aesx7gu33aNJeE9'}}]
